In [2]:
import os
os.chdir(r"D:\doc\2026\FINAL_CODE\src")
print(os.getcwd())

D:\doc\2026\FINAL_CODE\src


In [ ]:
import codeStat.modelimports.modelFunction as func
from codeStat.modelimports.modelFunction import *
import librosa
import glob
import pandas as pd

In [5]:
dir_save_path="D:/doc/2026/FINAL_CODE/src/codeStat/modelling/voiceModelling/srcVoiceModelling/GMM_TDNN/nonfix_out/GMM"

In [ ]:
# Load CSV sekali di luar loop
df_csv = pd.read_csv("D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/samplingStratified/out_samplingStratified/dataSampledSmall.csv")

# Buat dictionary: {audio_id: sentences}
dict_transkrip = {
    os.path.splitext(os.path.basename(path))[0]: kalimat
    for path, kalimat in zip(df_csv["Path_Data"], df_csv["sentences"])
}

def baca_transkrip(wav_path: str) -> str:
    """Ambil transkrip dari CSV berdasarkan ID file audio."""
    audio_id = os.path.splitext(os.path.basename(wav_path))[0]
    return dict_transkrip.get(audio_id, "").lower()

In [13]:
print(dict_transkrip)

{'00079b49f9': 'Hermes punika ugi dados utusanipun déwa Zeus', '002762f6a1': 'Lakhsmi garwanipun Wisnu punika minangka dewi kemakmuran saha kebahagiaan', '0034ac8d52': 'Kisah pepanggihan kalihan para Siren kacariyosaken ing kisah Odisseia', '00537fd57f': 'Muhammad Iqbal dolan menyang Cirebon', '005a3b3ad4': 'Stephen Chow lunga dhateng Finlandia', '0064a00864': 'Siti tansah eling sakjroning Enno Lerian', '00e1f837f4': 'Kristen Bell lunga dhateng Belfast', '011e017022': 'He moved at glacial speed towards the exam room', '0124f3cd94': 'Krabs rikala iku', '01a357ef3c': 'Rihanna lunga dhateng latvia', '01d15fd955': 'Saiki Kediri wis soyo maju ngoyak daerah liyane', '01e138192f': 'Wedhus kang biyasané ana ing kampung kampung ya iku wedhus kacukan', '01fbf0625b': 'Stephen Chow lunga dhateng Seoul', '020a55f3b6': 'Seogang utawi Kali Kulon mili saking sisih kulon', '020b073967': 'Emma Thompson kuwi alumni Cambridge University', '021d7997a3': 'Ronan Keating kuwi alumni University of Edinburgh', 

In [ ]:
# # ─────────────────────────────────────────────────────────────────────────────
# # KONFIGURASI
# # ─────────────────────────────────────────────────────────────────────────────
# sampled_root = func.sampled_small_root
# csv_path     = "D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/samplingStratified/out_samplingStratified/dataSampledSmall.csv"
# N_COMPONENTS = 16
# N_ITER       = 100

# # ─────────────────────────────────────────────────────────────────────────────
# # LOAD CSV — buat dictionary {audio_id: kalimat}
# # ─────────────────────────────────────────────────────────────────────────────

# df_csv = pd.read_csv(csv_path)

# dict_transkrip = {
#     os.path.splitext(os.path.basename(path))[0]: kalimat.lower()
#     for path, kalimat in zip(df_csv["Path_Data"], df_csv["sentences"])
# }

# print(f"Total transkrip dimuat: {len(dict_transkrip)} entri")

# def baca_transkrip(wav_path: str) -> str:
#     """Ambil kalimat dari dict berdasarkan ID file audio."""
#     audio_id = os.path.splitext(os.path.basename(wav_path))[0]
#     return dict_transkrip.get(audio_id, "")

# # ─────────────────────────────────────────────────────────────────────────────
# # TAHAP 1 — EKSTRAKSI FITUR MFCC
# # ─────────────────────────────────────────────────────────────────────────────

# hasil_mfcc  = []
# X_per_class = {}

# for wav_path in glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True):
#     try:
#         audio_id = os.path.splitext(os.path.basename(wav_path))[0]

#         fitur = extract_features(wav_path)   # (T, 39)
#         label = baca_transkrip(wav_path)

#         if label == "":          # skip jika tidak ada di CSV
#             continue

#         hasil_mfcc.append({
#             "Id"            : audio_id,
#             "Label"         : label,
#             "Jumlah Frame"  : fitur.shape[0],
#             "Dimensi Fitur" : fitur.shape[1],
#             "Contoh MFCC[0]": fitur[0, :3].tolist() if len(fitur) > 0 else [],
#         })

#         if label not in X_per_class:
#             X_per_class[label] = []
#         X_per_class[label].append(fitur)

#     except Exception as e:
#         print(f"Error pada file: {wav_path} : {e}")

# df_mfcc = pd.DataFrame(hasil_mfcc)
# df_mfcc.to_csv(f'{dir_save_path}/HasilMFCC.csv', index=False)
# print(f"HasilMFCC.csv     = {len(df_mfcc)} audio")
# print(f"Jumlah kelas unik = {len(X_per_class)}")

# # ─────────────────────────────────────────────────────────────────────────────
# # TAHAP 2 — PELATIHAN GMM PER KELAS
# # ─────────────────────────────────────────────────────────────────────────────

# X_per_class = {
#     label: np.vstack(frames)
#     for label, frames in X_per_class.items()
#     if len(frames) > 0
# }

# hasil_gmm_training = []
# gmm_models         = {}

# for label, X in X_per_class.items():
#     try:
#         gmm = GaussianMixtureModel(
#             n_components=N_COMPONENTS,
#             n_iter=N_ITER,
#             tol=1e-4,
#             reg_covar=1e-6,
#             random_state=hash(label) % 10000
#         )
#         gmm.fit(X)
#         gmm_models[label] = gmm

#         hasil_gmm_training.append({
#             "Label"               : label,
#             "Jumlah Bingkai"      : X.shape[0],
#             "Dimensi Fitur"       : X.shape[1],
#             "K Komponen"          : N_COMPONENTS,
#             "Iterasi EM"          : gmm.n_iter_done_,
#             "Konvergen"           : gmm.converged_,
#             "Log-Likelihood Akhir": round(gmm.log_likelihood_history_[-1], 6),
#             "Bobot w_k"           : gmm.weights_.round(4).tolist(),
#         })

#     except Exception as e:
#         print(f"Error pelatihan GMM label '{label}': {e}")

# df_gmm_training = pd.DataFrame(hasil_gmm_training)
# df_gmm_training.to_csv(f'{dir_save_path}/HasilGMMTraining.csv', index=False)
# print(f"HasilGMMTraining.csv = {len(df_gmm_training)} kelas")

# # ─────────────────────────────────────────────────────────────────────────────
# # TAHAP 3 — PREDIKSI & EVALUASI PER AUDIO
# # ─────────────────────────────────────────────────────────────────────────────

# hasil_prediksi   = []
# hasil_gmm_detail = []

# for wav_path in glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True):
#     try:
#         audio_id = os.path.splitext(os.path.basename(wav_path))[0]

#         fitur      = extract_features(wav_path)
#         label_asli = baca_transkrip(wav_path)

#         if label_asli == "":     # skip jika tidak ada di CSV
#             continue

#         skor_semua = {
#             label: float(gmm.score(fitur))
#             for label, gmm in gmm_models.items()
#         }

#         label_prediksi = max(skor_semua, key=skor_semua.get)
#         benar          = (label_prediksi == label_asli)

#         hasil_prediksi.append({
#             "Id"            : audio_id,
#             "Label Asli"    : label_asli,
#             "Label Prediksi": label_prediksi,
#             "Benar"         : benar,
#             "Skor Terbaik"  : round(skor_semua[label_prediksi], 4),
#         })

#         for kelas, skor in skor_semua.items():
#             hasil_gmm_detail.append({
#                 "Id"     : audio_id,
#                 "Kelas"  : kelas,
#                 "Skor"   : round(skor, 4),
#                 "Terbaik": (kelas == label_prediksi),
#             })

#     except Exception as e:
#         print(f"Error prediksi file: {wav_path} : {e}")

# df_prediksi   = pd.DataFrame(hasil_prediksi)
# df_gmm_detail = pd.DataFrame(hasil_gmm_detail)

# df_prediksi.to_csv(f'{dir_save_path}/HasilPrediksiGMM.csv', index=False)
# df_gmm_detail.to_csv(f'{dir_save_path}/HasilGMMDetail.csv', index=False)

# akurasi = df_prediksi["Benar"].mean() * 100 if len(df_prediksi) > 0 else 0
# print(f"HasilPrediksiGMM.csv = {len(df_prediksi)} audio")
# print(f"HasilGMMDetail.csv   = {len(df_gmm_detail)} baris")
# print(f"Akurasi Klasifikasi  = {akurasi:.2f}%")

Total transkrip dimuat: 1749 entri
HasilMFCC.csv     = 1749 audio
Jumlah kelas unik = 1725
    Iterasi   1: log-likelihood = -138.340271  Δ = inf
    Iterasi  10: log-likelihood = -124.848110  Δ = 7.50e-02
    Iterasi  20: log-likelihood = -124.586857  Δ = 7.40e-03
    Iterasi  30: log-likelihood = -124.525043  Δ = 1.61e-03
    Iterasi  40: log-likelihood = -124.483374  Δ = 3.68e-04
    Iterasi  50: log-likelihood = -124.466852  Δ = 2.52e-03
    ✓ Konvergen pada iterasi 54
    Iterasi   1: log-likelihood = -140.579324  Δ = inf
    Iterasi  10: log-likelihood = -128.088399  Δ = 6.99e-02
    Iterasi  20: log-likelihood = -127.851513  Δ = 1.06e-02
    Iterasi  30: log-likelihood = -127.650018  Δ = 6.04e-03
    Iterasi  40: log-likelihood = -127.583655  Δ = 2.69e-03
    Iterasi  50: log-likelihood = -127.573511  Δ = 6.16e-04
    Iterasi  60: log-likelihood = -127.564509  Δ = 9.72e-04
    Iterasi  70: log-likelihood = -127.535231  Δ = 1.27e-03
    ✓ Konvergen pada iterasi 77
    Iterasi   1

KeyboardInterrupt: 

In [ ]:
# # ─────────────────────────────────────────────────────────────────────────────
# # GMM GLOBAL — Acoustic Model Baseline untuk ASR (Pengenalan Wicara)
# #
# # Perubahan dari kode sebelumnya:
# #   SEBELUM : GMM per kelas (1 GMM per kalimat unik → 1749 model)
# #   SEKARANG: GMM global (1 GMM untuk semua frame MFCC)
# #
# # Alasan:
# #   - GMM dipakai sebagai acoustic model, bukan classifier kalimat
# #   - Sesuai sub-bab 2.2.9: GMM memodelkan distribusi MFCC secara keseluruhan
# #   - Log-likelihood bisa langsung dibandingkan dengan TDNN menggunakan WER
# # ─────────────────────────────────────────────────────────────────────────────

# import os
# import glob
# import numpy as np
# import pandas as pd

# # ─────────────────────────────────────────────────────────────────────────────
# # KONFIGURASI
# # ─────────────────────────────────────────────────────────────────────────────

# sampled_root = func.sampled_small_root
# csv_path     = "D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/samplingStratified/out_samplingStratified/dataSampledSmall.csv"
# N_COMPONENTS = 16    # jumlah komponen Gaussian
# N_ITER       = 100   # maksimum iterasi EM

# # ─────────────────────────────────────────────────────────────────────────────
# # LOAD CSV — buat dictionary {audio_id: kalimat}
# # ─────────────────────────────────────────────────────────────────────────────

# df_csv = pd.read_csv(csv_path)

# dict_transkrip = {
#     os.path.splitext(os.path.basename(path))[0]: kalimat.lower()
#     for path, kalimat in zip(df_csv["Path_Data"], df_csv["sentences"])
# }

# print(f"Total transkrip dimuat: {len(dict_transkrip)} entri")

# def baca_transkrip(wav_path: str) -> str:
#     """Ambil kalimat dari dict berdasarkan ID file audio."""
#     audio_id = os.path.splitext(os.path.basename(wav_path))[0]
#     return dict_transkrip.get(audio_id, "")


# # ─────────────────────────────────────────────────────────────────────────────
# # TAHAP 1 — EKSTRAKSI FITUR MFCC
# # Semua frame dari semua audio dikumpulkan ke satu pool global
# # ─────────────────────────────────────────────────────────────────────────────

# hasil_mfcc    = []
# semua_frame   = []   # <-- pool global, bukan per kelas
# meta_per_audio = []  # simpan (audio_id, label, jumlah_frame) untuk evaluasi

# for wav_path in glob.glob(os.path.join(sampled_root, "**/*.wav"), recursive=True):
#     try:
#         audio_id = os.path.splitext(os.path.basename(wav_path))[0]
#         fitur    = extract_features(wav_path)   # (T, 39)
#         label    = baca_transkrip(wav_path)

#         if label == "":
#             continue

#         # Kumpulkan semua frame ke pool global
#         semua_frame.append(fitur)

#         # Simpan metadata per audio untuk evaluasi
#         meta_per_audio.append({
#             "wav_path"  : wav_path,
#             "audio_id"  : audio_id,
#             "label"     : label,
#             "n_frames"  : fitur.shape[0],
#         })

#         hasil_mfcc.append({
#             "Id"            : audio_id,
#             "Label"         : label,
#             "Jumlah Frame"  : fitur.shape[0],
#             "Dimensi Fitur" : fitur.shape[1],
#             "Contoh MFCC[0]": fitur[0, :3].tolist() if len(fitur) > 0 else [],
#         })

#     except Exception as e:
#         print(f"Error pada file: {wav_path} : {e}")

# df_mfcc = pd.DataFrame(hasil_mfcc)
# df_mfcc.to_csv(f'{dir_save_path}/HasilMFCC.csv', index=False)
# print(f"HasilMFCC.csv       = {len(df_mfcc)} audio")
# print(f"Total frame global  = {sum(m['n_frames'] for m in meta_per_audio)}")


# # ─────────────────────────────────────────────────────────────────────────────
# # TAHAP 2 — PELATIHAN GMM GLOBAL
# # Satu GMM dilatih dari seluruh frame MFCC (bukan per kelas)
# # ─────────────────────────────────────────────────────────────────────────────

# # Gabungkan semua frame menjadi satu matriks besar (N_total_frame x 39)
# X_global = np.vstack(semua_frame)
# print(f"Shape X_global      = {X_global.shape}  "
#       f"(total frame x dimensi MFCC)")

# gmm_global = GaussianMixtureModel(
#     n_components=N_COMPONENTS,
#     n_iter=N_ITER,
#     tol=1e-4,
#     reg_covar=1e-6,
#     random_state=42
# )
# gmm_global.fit(X_global)

# # Simpan ringkasan pelatihan
# hasil_gmm_training = [{
#     "Model"               : "GMM Global",
#     "Jumlah Audio"        : len(meta_per_audio),
#     "Total Frame"         : X_global.shape[0],
#     "Dimensi Fitur"       : X_global.shape[1],
#     "K Komponen"          : N_COMPONENTS,
#     "Iterasi EM"          : gmm_global.n_iter_done_,
#     "Konvergen"           : gmm_global.converged_,
#     "Log-Likelihood Akhir": round(gmm_global.log_likelihood_history_[-1], 6),
#     "Bobot w_k"           : gmm_global.weights_.round(4).tolist(),
# }]

# df_gmm_training = pd.DataFrame(hasil_gmm_training)
# df_gmm_training.to_csv(f'{dir_save_path}/HasilGMMTraining.csv', index=False)
# print(f"\n── Hasil Pelatihan GMM Global ──")
# print(f"  Konvergen     : {gmm_global.converged_}")
# print(f"  Iterasi EM    : {gmm_global.n_iter_done_}")
# print(f"  Log-Likelihood: {gmm_global.log_likelihood_history_[-1]:.4f}")


# # ─────────────────────────────────────────────────────────────────────────────
# # TAHAP 3 — EVALUASI PER AUDIO
# #
# # Karena GMM global bukan classifier kalimat, evaluasi dilakukan dengan:
# #   - Log-likelihood per audio  → ukuran kualitas akustik
# #   - Log-likelihood per frame  → rata-rata per frame (normalized)
# #
# # Metrik utama untuk perbandingan dengan TDNN adalah WER (Word Error Rate).
# # Log-likelihood di sini dipakai sebagai diagnostic / sanity check akustik.
# # ─────────────────────────────────────────────────────────────────────────────

# hasil_evaluasi    = []
# semua_ll_per_frame = []

# for meta in meta_per_audio:
#     try:
#         fitur = extract_features(meta["wav_path"])   # (T, 39)

#         # gmm.score() = rata-rata log-likelihood per frame
#         ll_per_frame = float(gmm_global.score(fitur))
#         ll_total     = ll_per_frame * fitur.shape[0]

#         semua_ll_per_frame.append(ll_per_frame)

#         hasil_evaluasi.append({
#             "Id"                  : meta["audio_id"],
#             "Label"               : meta["label"],
#             "Jumlah Frame"        : fitur.shape[0],
#             "LL per Frame"        : round(ll_per_frame, 4),
#             "LL Total"            : round(ll_total, 4),
#         })

#     except Exception as e:
#         print(f"Error evaluasi file: {meta['wav_path']} : {e}")

# df_evaluasi = pd.DataFrame(hasil_evaluasi)
# df_evaluasi.to_csv(f'{dir_save_path}/HasilEvaluasiGMM.csv', index=False)

# # ─────────────────────────────────────────────────────────────────────────────
# # RINGKASAN STATISTIK
# # ─────────────────────────────────────────────────────────────────────────────

# ll_arr = np.array(semua_ll_per_frame)

# print(f"\n── Statistik Log-Likelihood per Frame ──")
# print(f"  Mean   : {ll_arr.mean():.4f}")
# print(f"  Std    : {ll_arr.std():.4f}")
# print(f"  Min    : {ll_arr.min():.4f}")
# print(f"  Max    : {ll_arr.max():.4f}")
# print(f"\nHasilEvaluasiGMM.csv = {len(df_evaluasi)} audio")
# print(
#     "\nCatatan: Untuk perbandingan dengan TDNN, gunakan WER (Word Error Rate).\n"
#     "Log-likelihood GMM global di sini berfungsi sebagai baseline akustik,\n"
#     "bukan sebagai classifier kalimat."
# )

Total transkrip dimuat: 1749 entri
HasilMFCC.csv       = 1749 audio
Total frame global  = 972792
Shape X_global      = (972792, 39)  (total frame x dimensi MFCC)
    Iterasi   1: log-likelihood = -143.900973  Δ = inf
    Iterasi  10: log-likelihood = -132.695967  Δ = 2.99e-02
    Iterasi  20: log-likelihood = -132.582437  Δ = 4.33e-03
    Iterasi  30: log-likelihood = -132.554026  Δ = 2.60e-03
    Iterasi  40: log-likelihood = -132.528025  Δ = 1.80e-03
    Iterasi  50: log-likelihood = -132.518578  Δ = 6.15e-04
    Iterasi  60: log-likelihood = -132.511300  Δ = 9.12e-04
    Iterasi  70: log-likelihood = -132.502852  Δ = 6.01e-04
    Iterasi  80: log-likelihood = -132.499886  Δ = 1.36e-04
    ✓ Konvergen pada iterasi 83

── Hasil Pelatihan GMM Global ──
  Konvergen     : True
  Iterasi EM    : 83
  Log-Likelihood: -132.4996

── Statistik Log-Likelihood per Frame ──
  Mean   : -132.5348
  Std    : 2.7237
  Min    : -147.2974
  Max    : -123.9784

HasilEvaluasiGMM.csv = 1749 audio

Catata

In [ ]:
"""
main_gmm.py
===========
Entry point pipeline GMM ASR Bahasa Jawa — versi EFISIEN.

Cara jalankan:
    python main_gmm.py

Urutan eksekusi:
  [1] Load transkrip CSV
  [2] Extract MFCC semua audio → cache (SEKALI SAJA)
  [3] Kumpulkan frame per kata dari cache (SEKALI SAJA)
  [4] Latih GMM global (opsional, untuk perbandingan)
  [5] Latih GMM per kata
  [6] Latih N-Gram Trigram
  [7] Dekode + hitung WER
  [8] Hyperparameter tuning (hanya re-train GMM + re-decode)
  [9] Simpan semua hasil CSV

Output files:
  HasilMFCC.csv                       — info frame tiap audio
  HasilGMMGlobal.csv                  — info GMM global
  HasilGMMPerKata.csv                 — info GMM per kata
  HasilWER_GMM.csv                    — WER detail tiap audio
  HasilRingkasanWER.csv               — ringkasan statistik WER
  HasilHyperparameterTuning_GMM.csv   — hasil grid search
"""

import os
import time
import numpy as np
import pandas as pd

# ══════════════════════════════════════════════════════════════════════════════
# ★ KONFIGURASI — Sesuaikan path di sini
# ══════════════════════════════════════════════════════════════════════════════

# Path root folder audio (WAV/FLAC)
SAMPLED_ROOT = r"D:/doc/2026/FINAL_CODE/src/data/sampled_small"

# Path CSV transkrip hasil stratified sampling
CSV_PATH = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/"
    r"samplingStratified/out_samplingStratified/dataSampledSmall.csv"
)

# Folder output hasil
DIR_SAVE = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/modelling/"
    r"voiceModelling/srcVoiceModelling/GMM_TDNN/out/GMM"
)

# ── Hyperparameter utama (sesuai sub-bab 3.3.5 skripsi) ──────────────────────
N_COMPONENTS = 16      # K komponen GMM per kata (akan di-tune juga)
N_ITER       = 100     # maks iterasi EM
TOL          = 1e-4    # toleransi konvergensi
N_GRAM       = 3       # orde N-gram (trigram, sesuai sub-bab 3.3.7)
TOP_K_DECODE = 10      # top-K kandidat kata untuk N-gram rescore

# ── Flag eksekusi ─────────────────────────────────────────────────────────────
JALANKAN_GLOBAL_GMM = True    # latih GMM global (opsional)
JALANKAN_TUNING     = True    # jalankan hyperparameter tuning


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    os.makedirs(DIR_SAVE, exist_ok=True)
    t_start = time.time()

    print("=" * 65)
    print("  GMM ASR BAHASA JAWA — Pipeline Efisien")
    print("=" * 65)

    # ────────────────────────────────────────────────────────────────────────
    # TAHAP 1 — LOAD TRANSKRIP
    # ────────────────────────────────────────────────────────────────────────
    print("\n[TAHAP 1] Membaca transkrip ...")
    dict_transkrip  = baca_transkrip_csv(CSV_PATH)
    semua_transkrip = list(dict_transkrip.values())
    print(f"  Total audio dengan transkrip: {len(dict_transkrip)}")

    lexicon = buat_lexicon(semua_transkrip)
    print(f"  Vocab kata unik : {len(lexicon)}")
    pd.DataFrame([
        {"Kata": k, "Fonem": " ".join(v)} for k, v in lexicon.items()
    ]).to_csv(f"{DIR_SAVE}/HasilLexicon.csv", index=False)


    # ────────────────────────────────────────────────────────────────────────
    # TAHAP 2 — EXTRACT MFCC → CACHE (KUNCI OPTIMASI)
    # Hanya dipanggil SEKALI. Semua tahap berikutnya pakai cache ini.
    # ────────────────────────────────────────────────────────────────────────
    print("\n[TAHAP 2] Ekstraksi MFCC (sekali saja) ...")
    cache_mfcc, meta_list = build_mfcc_cache(
        sampled_root    = SAMPLED_ROOT,
        dict_transkrip  = dict_transkrip,
        verbose         = True,
    )
    df_mfcc = pd.DataFrame(meta_list)
    df_mfcc.to_csv(f"{DIR_SAVE}/HasilMFCC.csv", index=False)
    print(f"  HasilMFCC.csv → {len(df_mfcc)} audio")

    if len(cache_mfcc) == 0:
        print("  [ERROR] Tidak ada audio yang berhasil di-load. Cek path SAMPLED_ROOT.")
        return


    # ────────────────────────────────────────────────────────────────────────
    # TAHAP 3 — GMM GLOBAL (opsional)
    # ────────────────────────────────────────────────────────────────────────
    if JALANKAN_GLOBAL_GMM:
        print("\n[TAHAP 3] Melatih GMM Global ...")
        X_global = np.vstack(list(cache_mfcc.values()))
        print(f"  Shape X_global: {X_global.shape}")

        gmm_global = GaussianMixtureModel(
            n_components = N_COMPONENTS,
            n_iter       = N_ITER,
            tol          = TOL,
            reg_covar    = 1e-6,
            random_state = 42,
        )
        gmm_global.fit(X_global)

        pd.DataFrame([{
            "K Komponen"          : N_COMPONENTS,
            "Iterasi EM"          : gmm_global.n_iter_done_,
            "Konvergen"           : gmm_global.converged_,
            "Log-Likelihood Akhir": round(gmm_global.log_likelihood_history_[-1], 6),
        }]).to_csv(f"{DIR_SAVE}/HasilGMMGlobal.csv", index=False)

        print(f"  Konvergen     : {gmm_global.converged_}")
        print(f"  Iterasi EM    : {gmm_global.n_iter_done_}")
        print(f"  Log-Likelihood: {gmm_global.log_likelihood_history_[-1]:.4f}")
    else:
        print("\n[TAHAP 3] GMM Global dilewati (JALANKAN_GLOBAL_GMM=False)")


    # ────────────────────────────────────────────────────────────────────────
    # TAHAP 4 — KUMPULKAN FRAME PER KATA (dari cache, bukan baca file)
    # ────────────────────────────────────────────────────────────────────────
    print("\n[TAHAP 4] Mengumpulkan frame per kata (dari cache) ...")
    frame_per_kata = kumpulkan_frame_per_kata_dari_cache(
        cache_mfcc    = cache_mfcc,
        dict_transkrip= dict_transkrip,
        verbose       = True,
    )


    # ────────────────────────────────────────────────────────────────────────
    # TAHAP 5 — LATIH GMM PER KATA
    # ────────────────────────────────────────────────────────────────────────
    print(f"\n[TAHAP 5] Melatih GMM per kata (K={N_COMPONENTS}) ...")
    gmm_per_kata = latih_gmm_per_kata(
        frame_per_kata = frame_per_kata,
        n_components   = N_COMPONENTS,
        n_iter         = N_ITER,
        tol            = TOL,
        verbose        = True,
    )

    # Simpan ringkasan training GMM per kata
    hasil_training = []
    for kata, gmm in gmm_per_kata.items():
        frame_total = np.vstack(frame_per_kata[kata]).shape[0]
        hasil_training.append({
            "Kata"                : kata,
            "Jumlah Frame"        : frame_total,
            "K Komponen"          : gmm.n_components,
            "Iterasi EM"          : gmm.n_iter_done_,
            "Konvergen"           : gmm.converged_,
            "Log-Likelihood Akhir": round(gmm.log_likelihood_history_[-1], 4),
        })
    pd.DataFrame(hasil_training).to_csv(
        f"{DIR_SAVE}/HasilGMMPerKata.csv", index=False)
    print(f"  HasilGMMPerKata.csv → {len(hasil_training)} kata")


    # ────────────────────────────────────────────────────────────────────────
    # TAHAP 6 — LATIH N-GRAM LANGUAGE MODEL
    # ────────────────────────────────────────────────────────────────────────
    print(f"\n[TAHAP 6] Melatih {N_GRAM}-Gram Language Model ...")
    ngram_model = NGramModel(n=N_GRAM, discount=0.75)
    ngram_model.fit(semua_transkrip)
    print(f"  Vocab size : {len(ngram_model.vocab)}")
    print(f"  N-gram unik: {len(ngram_model.ngram_count)}")


    # ────────────────────────────────────────────────────────────────────────
    # TAHAP 7 — DEKODE + WER (dari cache, tidak baca file)
    # ────────────────────────────────────────────────────────────────────────
    print(f"\n[TAHAP 7] Dekode audio + hitung WER ...")
    hasil_wer = dekode_semua_audio(
        cache_mfcc    = cache_mfcc,
        dict_transkrip= dict_transkrip,
        gmm_per_kata  = gmm_per_kata,
        ngram_model   = ngram_model,
        top_k         = TOP_K_DECODE,
        verbose       = True,
    )

    df_wer = pd.DataFrame(hasil_wer)
    df_wer.to_csv(f"{DIR_SAVE}/HasilWER_GMM.csv", index=False)
    print(f"  HasilWER_GMM.csv → {len(df_wer)} audio")

    stat_wer = ringkasan_wer(df_wer, label="GMM per Kata")
    pd.DataFrame([stat_wer]).to_csv(f"{DIR_SAVE}/HasilRingkasanWER.csv", index=False)


    # ────────────────────────────────────────────────────────────────────────
    # TAHAP 8 — HYPERPARAMETER TUNING
    # frame_per_kata & cache_mfcc sudah ada → hanya re-train GMM + re-decode
    # Estimasi hemat waktu: dari ~24 × pipeline penuh → 24 × (train + decode) saja
    # ────────────────────────────────────────────────────────────────────────
    if JALANKAN_TUNING:
        print(f"\n[TAHAP 8] Hyperparameter Tuning ...")
        df_tuning = hyperparameter_tuning(
            frame_per_kata  = frame_per_kata,   # ← sudah ada, tidak dikumpulkan ulang
            cache_mfcc      = cache_mfcc,       # ← sudah ada, tidak di-extract ulang
            dict_transkrip  = dict_transkrip,
            ngram_model     = ngram_model,
            dir_save        = DIR_SAVE,
            n_gram          = N_GRAM,
        )
    else:
        print("\n[TAHAP 8] Tuning dilewati (JALANKAN_TUNING=False)")


    # ────────────────────────────────────────────────────────────────────────
    # SELESAI
    # ────────────────────────────────────────────────────────────────────────
    total_time = time.time() - t_start
    print("\n" + "=" * 65)
    print(f"  SELESAI dalam {total_time/60:.1f} menit")
    print(f"  Output tersimpan di: {DIR_SAVE}")
    print("=" * 65)
    print(f"  HasilLexicon.csv                     ✓")
    print(f"  HasilMFCC.csv                        ✓")
    if JALANKAN_GLOBAL_GMM:
        print(f"  HasilGMMGlobal.csv                   ✓")
    print(f"  HasilGMMPerKata.csv                  ✓")
    print(f"  HasilWER_GMM.csv                     ✓")
    print(f"  HasilRingkasanWER.csv                ✓")
    if JALANKAN_TUNING:
        print(f"  HasilHyperparameterTuning_GMM.csv   ✓")


if __name__ == "__main__":
    main()